# Consultas de prueba · DW modelo estrella (`dw.*`)

**Objetivo:** verificar que el modelo estrella responde preguntas analíticas reales,
y servir de evidencia para la defensa. Cada consulta hace un **JOIN del hecho con sus
dimensiones** — así se comprueba que la estrella funciona de punta a punta.

> Patrón Kimball: el hecho (`fact_defunciones`) tiene las llaves; las descripciones
> (nombre de mes, grupo de edad, etnia, capítulo CIE-10) viven en las dimensiones.
> Por eso todo análisis es `fact JOIN dim`.

Ejecuta una celda a la vez y lee el resultado.

In [ ]:
spark.sql("USE CATALOG workspace")

## 1. Sanity check — total de defunciones por año

In [ ]:
# 1. Defunciones por año (fact JOIN dim_tiempo)
spark.sql("""
SELECT t.anio,
       SUM(f.cantidad) AS defunciones
FROM dw.fact_defunciones f
JOIN dw.dim_tiempo t ON f.id_tiempo = t.id_tiempo
GROUP BY t.anio
ORDER BY t.anio
""").show()

## 2. Top 10 causas de muerte (con descripción de capítulo CIE-10)

In [ ]:
# 2. Top 10 causas CIE-10
spark.sql("""
SELECT c.codigo_completo,
       c.capitulo_1,
       c.mal_definida,
       SUM(f.cantidad) AS muertes
FROM dw.fact_defunciones f
JOIN dw.dim_causa_cie10 c ON f.id_causa = c.id_causa
GROUP BY c.codigo_completo, c.capitulo_1, c.mal_definida
ORDER BY muertes DESC
LIMIT 10
""").show()

## 3. Pre vs Post-COVID
Compara defunciones PRE_COVID (2015-2019) vs POST_COVID (2020-2024).

In [ ]:
# 3. Pre vs Post COVID
spark.sql("""
SELECT f.periodo,
       SUM(f.cantidad)                  AS defunciones,
       COUNT(DISTINCT t.anio)           AS anios,
       ROUND(SUM(f.cantidad) / COUNT(DISTINCT t.anio), 0) AS promedio_anual
FROM dw.fact_defunciones f
JOIN dw.dim_tiempo t ON f.id_tiempo = t.id_tiempo
GROUP BY f.periodo
ORDER BY f.periodo
""").show()

## 4. COVID-19 por año

In [ ]:
# 4. Muertes por COVID-19 (U071) por año
spark.sql("""
SELECT t.anio,
       SUM(f.cantidad) AS muertes_covid
FROM dw.fact_defunciones f
JOIN dw.dim_tiempo t       ON f.id_tiempo = t.id_tiempo
JOIN dw.dim_causa_cie10 c  ON f.id_causa  = c.id_causa
WHERE c.codigo_completo = 'U071'
GROUP BY t.anio
ORDER BY t.anio
""").show()

## 5. Distribución por grupo de edad y sexo

In [ ]:
# 5. Defunciones por grupo etario y sexo
spark.sql("""
SELECT g.grupo_edad,
       s.sexo_desc,
       SUM(f.cantidad) AS defunciones
FROM dw.fact_defunciones f
JOIN dw.dim_grupo_etario g ON f.id_grupo_etario = g.id_grupo_etario
JOIN dw.dim_sexo s         ON f.id_sexo = s.id_sexo
GROUP BY g.grupo_edad, s.sexo_desc
ORDER BY defunciones DESC
""").show(20)

## 6. Top departamentos y mortalidad infantil (<1 año)

In [ ]:
# 6. Mortalidad infantil (<1 año) por departamento — top 10
spark.sql("""
SELECT g.codigo_depto,
       SUM(f.cantidad) AS muertes_menores_1
FROM dw.fact_defunciones f
JOIN dw.dim_geografia g     ON f.id_geografia = g.id_geografia
JOIN dw.dim_grupo_etario ge ON f.id_grupo_etario = ge.id_grupo_etario
WHERE ge.grupo_edad = '< 1 año'
GROUP BY g.codigo_depto
ORDER BY muertes_menores_1 DESC
LIMIT 10
""").show()

## 7. Calidad del registro — % de causas mal definidas

In [ ]:
# 7. % de causas mal definidas
spark.sql("""
SELECT
    SUM(CASE WHEN c.mal_definida THEN f.cantidad ELSE 0 END) AS mal_definidas,
    SUM(f.cantidad)                                          AS total,
    ROUND(100.0 * SUM(CASE WHEN c.mal_definida THEN f.cantidad ELSE 0 END)
          / SUM(f.cantidad), 2)                              AS pct_mal_definidas
FROM dw.fact_defunciones f
JOIN dw.dim_causa_cie10 c ON f.id_causa = c.id_causa
""").show()